In [0]:
from pyspark.sql import SparkSession

In [0]:
spark = SparkSession.builder.appName("E2E_Pipeline").getOrCreate()

In [0]:
data = [
   (1, "C001", "Laptop", "50000", "2024-01-01"),
   (2, "C002", "Mobile", None, "2024-01-02"),
   (3, "C003", "Tablet", "20000", "2024-01-03"),
   (4, "C004", "Laptop", "55000", "2024-01-04"),
   (5, "C005", "Headphones", None, "2024-01-05"),
   ##(3, "C003", "Tablet", "22000", "2024-01-06"),  # updated
   (6, "C006", "Camera", "30000", "2024-01-06"),
   (7, "C007", "Mobile", "18000", "2024-01-07"),
   (8, "C008", "Watch", "8000", "2024-01-07")]

In [0]:
columns = ["order_id", "customer_id", "product", "amount", "updated_date"]


In [0]:
df = spark.createDataFrame(data, columns)

In [0]:
df.display()

order_id,customer_id,product,amount,updated_date
1,C001,Laptop,50000,2024-01-01
2,C002,Mobile,null,2024-01-02
3,C003,Tablet,20000,2024-01-03
4,C004,Laptop,55000,2024-01-04
5,C005,Headphones,null,2024-01-05
6,C006,Camera,30000,2024-01-06
7,C007,Mobile,18000,2024-01-07
8,C008,Watch,8000,2024-01-07


In [0]:
df=df.fillna({"amount":0})


In [0]:
df=df.fillna({"amount":"0"})

In [0]:
df.display()

order_id,customer_id,product,amount,updated_date
1,C001,Laptop,50000,2024-01-01
2,C002,Mobile,0,2024-01-02
3,C003,Tablet,20000,2024-01-03
4,C004,Laptop,55000,2024-01-04
5,C005,Headphones,0,2024-01-05
6,C006,Camera,30000,2024-01-06
7,C007,Mobile,18000,2024-01-07
8,C008,Watch,8000,2024-01-07


In [0]:
df=df.replace("0","1000",subset=["amount"])

In [0]:
df.display()

order_id,customer_id,product,amount,updated_date
1,C001,Laptop,50000,2024-01-01
2,C002,Mobile,1000,2024-01-02
3,C003,Tablet,20000,2024-01-03
4,C004,Laptop,55000,2024-01-04
5,C005,Headphones,1000,2024-01-05
6,C006,Camera,30000,2024-01-06
7,C007,Mobile,18000,2024-01-07
8,C008,Watch,8000,2024-01-07


In [0]:
from pyspark.sql.functions import col
df=df.withColumn("amount",col("amount").cast("int"))\
    .withColumn("updated_date",col("updated_date").cast("date"))

In [0]:
df=df.withColumnRenamed("updated_date","date")

In [0]:
df.display()

order_id,customer_id,product,amount,date
1,C001,Laptop,50000,2024-01-01
2,C002,Mobile,1000,2024-01-02
3,C003,Tablet,20000,2024-01-03
4,C004,Laptop,55000,2024-01-04
5,C005,Headphones,1000,2024-01-05
6,C006,Camera,30000,2024-01-06
7,C007,Mobile,18000,2024-01-07
8,C008,Watch,8000,2024-01-07


In [0]:
df=df.withColumn("bonus",col("amount")*0.1)
df.display()

order_id,customer_id,product,amount,date,bonus
1,C001,Laptop,50000,2024-01-01,5000.0
2,C002,Mobile,1000,2024-01-02,100.0
3,C003,Tablet,20000,2024-01-03,2000.0
4,C004,Laptop,55000,2024-01-04,5500.0
5,C005,Headphones,1000,2024-01-05,100.0
6,C006,Camera,30000,2024-01-06,3000.0
7,C007,Mobile,18000,2024-01-07,1800.0
8,C008,Watch,8000,2024-01-07,800.0


In [0]:
from pyspark.sql.functions import when
df=df.withColumn(
    "category", when(col("amount")>20000,"high").otherwise("low"))
df.display()

order_id,customer_id,product,amount,date,bonus,category
1,C001,Laptop,50000,2024-01-01,5000.0,high
2,C002,Mobile,1000,2024-01-02,100.0,low
3,C003,Tablet,20000,2024-01-03,2000.0,low
4,C004,Laptop,55000,2024-01-04,5500.0,high
5,C005,Headphones,1000,2024-01-05,100.0,low
6,C006,Camera,30000,2024-01-06,3000.0,high
7,C007,Mobile,18000,2024-01-07,1800.0,low
8,C008,Watch,8000,2024-01-07,800.0,low


In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
def amount_bucket_function(amount):
    if amount<10000:
        return "low"
    elif amount<=30000:
        return "mid"
    else:
        return "High"
    
amount_bucket_udf=udf(amount_bucket_function,StringType())
df = df.withColumn("category", amount_bucket_udf(col("amount")))
df.display()





order_id,customer_id,product,amount,date,bonus,category
1,C001,Laptop,50000,2024-01-01,5000.0,High
2,C002,Mobile,1000,2024-01-02,100.0,low
3,C003,Tablet,20000,2024-01-03,2000.0,mid
4,C004,Laptop,55000,2024-01-04,5500.0,High
5,C005,Headphones,1000,2024-01-05,100.0,low
6,C006,Camera,30000,2024-01-06,3000.0,mid
7,C007,Mobile,18000,2024-01-07,1800.0,mid
8,C008,Watch,8000,2024-01-07,800.0,low


In [0]:
df.write.mode("overwrite").saveAsTable("bronze")

In [0]:
from pyspark.sql.functions import col
last_loaded_date="2024-01-03"
df_incremental=df.filter(col("date")>last_loaded_date)


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
window_spec=Window.partitionBy("order_id").orderBy(col("date").desc())
df_dedup=df_incremental.withColumn("rn",row_number().over(window_spec)).filter(col("rn")==1).drop("rn")


In [0]:
dbutils.widgets.text("input_path", "/mnt/data/source/")
dbutils.widgets.text("last_loaded_date", "2024-01-01")

input_path = dbutils.widgets.get("input_path")
last_loaded_date = dbutils.widgets.get("last_loaded_date")

In [0]:
df.display()

order_id,customer_id,product,amount,date,bonus,category
1,C001,Laptop,50000,2024-01-01,5000.0,High
2,C002,Mobile,1000,2024-01-02,100.0,low
3,C003,Tablet,20000,2024-01-03,2000.0,mid
4,C004,Laptop,55000,2024-01-04,5500.0,High
5,C005,Headphones,1000,2024-01-05,100.0,low
6,C006,Camera,30000,2024-01-06,3000.0,mid
7,C007,Mobile,18000,2024-01-07,1800.0,mid
8,C008,Watch,8000,2024-01-07,800.0,low
